# DOCX parsing for code documentation

Convert all DOCX files below `assets/code-docs` to Markdown while retaining numbered headings and complete HTML tables, including merged cells and nested tables. Embedded images are extracted to relative `img/` links.

Run this notebook with the project’s `.venv` Python kernel (`uv sync` installs the dependencies). No Word, LibreOffice, external API, or nbdev export is required. Source documents are never changed.


## Dependencies

Mammoth reads Word documents; Markdownify converts surrounding content to Markdown. `tqdm.auto` selects notebook widgets when available and text progress otherwise.

In [ ]:
import hashlib
from html import escape
import io
import json
import mimetypes
import os
import re
import shutil
import tempfile
from copy import deepcopy
from importlib.metadata import version
from pathlib import Path
from zipfile import ZipFile

import mammoth
from bs4 import BeautifulSoup, NavigableString, Tag
from bs4.formatter import HTMLFormatter
from lxml import etree as ET
from markdownify import MarkdownConverter
from tqdm.auto import tqdm


## Preserve Word section enumerations

Word often stores a heading such as **1.2 使用者定义** as text **使用者定义** plus automatic numbering inherited from a style. The following helpers resolve styles, list instances, level templates, starts, overrides, and restarts in document order. They materialize the heading labels only in an in-memory DOCX copy and explicitly map custom heading styles to HTML headings. Existing literal copies of the calculated label are not duplicated. Unsupported automatic heading numbering fails that file visibly rather than dropping the enumeration.

Heading levels 7–9 are represented as Markdown level 6, the deepest HTML/Markdown heading level. Body lists stay with Mammoth’s normal list conversion; exact Word pagination and visual styling are outside scope.
Inherited subsection numbering follows the active parent list instance when both use the same abstract outline; explicit paragraph numbering and restart overrides remain independent.


In [ ]:
W = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}"
XML_SPACE = "{http://www.w3.org/XML/1998/namespace}space"
PIPELINE_VERSION = 3


def _xml(data):
    return ET.fromstring(data, parser=ET.XMLParser(resolve_entities=False, no_network=True))


def _val(element, path, default=None):
    node = element.find(path) if element is not None else None
    return node.get(W + "val", default) if node is not None else default


def _style_chain(style_id, styles):
    """Return base-to-derived styles; reject cyclic inheritance."""
    chain, seen = [], set()
    while style_id and style_id in styles:
        if style_id in seen:
            raise ValueError(f"Cyclic Word style inheritance: {style_id}")
        seen.add(style_id)
        style = styles[style_id]
        chain.append(style)
        style_id = _val(style, W + "basedOn")
    return list(reversed(chain))


def _paragraph_properties(paragraph, styles):
    ppr = paragraph.find(W + "pPr")
    style_id = _val(ppr, W + "pStyle")
    chain = _style_chain(style_id, styles)
    properties = [s.find(W + "pPr") for s in chain] + [ppr]
    num_id, level, outline = None, None, None
    for prop in properties:
        num_id = _val(prop, W + "numPr/" + W + "numId", num_id)
        level = _val(prop, W + "numPr/" + W + "ilvl", level)
        outline = _val(prop, W + "outlineLvl", outline)
    if outline is not None:
        heading = min(int(outline) + 1, 6) if int(outline) < 9 else None
    else:
        heading = None
        for style in reversed(chain):
            name = _val(style, W + "name", "")
            match = re.fullmatch(r"(?:heading|标题)\s*([1-9])", name, re.I)
            if match:
                heading = min(int(match[1]), 6)
                break
    return style_id, num_id, int(level) if level is not None else None, heading


def _format_number(number, fmt):
    if fmt == "decimal":
        return str(number)
    if fmt == "decimalZero":
        return f"{number:02d}"
    if fmt in {"lowerLetter", "upperLetter"}:
        if number < 1:
            raise ValueError("Letter numbering requires a positive counter")
        text = ""
        while number:
            number, rem = divmod(number - 1, 26)
            text = chr(65 + rem) + text
        return text.lower() if fmt == "lowerLetter" else text
    if fmt in {"lowerRoman", "upperRoman"}:
        if not 0 < number < 4000:
            raise ValueError("Roman heading numbering outside 1..3999")
        text = ""
        for amount, symbol in [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
                               (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
                               (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]:
            count, number = divmod(number, amount)
            text += symbol * count
        return text.lower() if fmt == "lowerRoman" else text
    if fmt in {"chineseCounting", "chineseCountingThousand", "japaneseCounting"}:
        if not 0 <= number <= 9999:
            raise ValueError("Chinese/Japanese heading numbering outside 0..9999")
        if number == 0:
            return "〇" if fmt == "japaneseCounting" else "零"
        digits = "零一二三四五六七八九"
        text, pending_zero = "", False
        for divisor, unit in [(1000, "千"), (100, "百"), (10, "十"), (1, "")]:
            digit, number = divmod(number, divisor)
            if digit:
                if pending_zero and fmt != "japaneseCounting":
                    text += "零"
                if not (digit == 1 and unit == "十" and not text) and not (
                    fmt == "japaneseCounting" and digit == 1 and unit
                ):
                    text += digits[digit]
                text += unit
                pending_zero = False
            elif text and number:
                pending_zero = True
        return text
    if fmt == "none":
        return ""
    raise ValueError(f"Unsupported automatic heading number format: {fmt}")


class _WordNumbering:
    """Resolve Word list instances, style links, overrides, and heading counters."""

    def __init__(self, root, styles):
        self.styles = styles
        self.abstract = {e.get(W + "abstractNumId"): e for e in root.findall(W + "abstractNum")}
        self.nums = {e.get(W + "numId"): e for e in root.findall(W + "num")}
        self.cache, self.counters = {}, {}

    def levels(self, num_id, seen=()):
        if num_id in self.cache:
            return self.cache[num_id]
        if num_id in seen or num_id not in self.nums:
            raise ValueError(f"Unresolved/cyclic Word numbering instance: {num_id}")
        num = self.nums[num_id]
        abstract_id = _val(num, W + "abstractNumId")
        if abstract_id not in self.abstract:
            raise ValueError(f"Missing Word abstract numbering: {abstract_id}")
        abstract = self.abstract[abstract_id]
        linked = _val(abstract, W + "numStyleLink")
        if linked:
            linked_num = None
            for style in _style_chain(linked, self.styles):
                linked_num = _val(style, W + "pPr/" + W + "numPr/" + W + "numId", linked_num)
            levels = deepcopy(self.levels(linked_num, (*seen, num_id)))
        else:
            levels = {int(e.get(W + "ilvl", "0")): deepcopy(e) for e in abstract.findall(W + "lvl")}
        for override in num.findall(W + "lvlOverride"):
            index = int(override.get(W + "ilvl"))
            replacement = override.find(W + "lvl")
            if replacement is not None:
                levels[index] = deepcopy(replacement)
            start = _val(override, W + "startOverride")
            if start is not None:
                if index not in levels:
                    raise ValueError(f"Start override has no numbering level: {num_id}/{index}")
                node = levels[index].find(W + "start")
                if node is None:
                    node = ET.SubElement(levels[index], W + "start")
                node.set(W + "val", start)
        self.cache[num_id] = levels
        return levels

    def advance(self, num_id, level, style_id):
        levels = self.levels(num_id)
        if level is None:
            matching = [i for i, node in levels.items() if _val(node, W + "pStyle") == style_id]
            level = matching[0] if len(matching) == 1 else 0
        if level not in levels:
            raise ValueError(f"Missing Word numbering level: {num_id}/{level}")
        counts = self.counters.setdefault(num_id, {})
        for child, node in levels.items():
            # 0 means never restart; otherwise Word uses a one-based ancestor level.
            restart = int(_val(node, W + "lvlRestart", str(child)))
            if child > level and restart and level < restart:
                counts.pop(child, None)
        start = int(_val(levels[level], W + "start", "1"))
        counts[level] = counts[level] + 1 if level in counts else start
        return level

    def label(self, num_id, level):
        levels = self.levels(num_id)
        node = levels[level]
        fmt = _val(node, W + "numFmt", "decimal")
        template = _val(node, W + "lvlText")
        if template is None:
            raise ValueError(f"Missing heading numbering template: {num_id}/{level}")
        if fmt == "bullet":
            return template
        legal_node = node.find(W + "isLgl")
        legal = legal_node is not None and legal_node.get(W + "val", "1") not in {"0", "false", "off"}

        def substitute(match):
            index = int(match.group(1)) - 1
            if index not in levels:
                raise ValueError(f"Heading label references missing level {index}")
            value = self.counters[num_id].get(index, int(_val(levels[index], W + "start", "1")))
            number_format = "decimal" if legal else _val(levels[index], W + "numFmt", "decimal")
            return _format_number(value, number_format)

        return re.sub(r"%([1-9])", substitute, template)


def prepare_numbered_docx(source_bytes):
    """Materialize heading labels in a new in-memory archive; never edit the input."""
    with ZipFile(io.BytesIO(source_bytes)) as archive:
        document = _xml(archive.read("word/document.xml"))
        styles_root = _xml(archive.read("word/styles.xml")) if "word/styles.xml" in archive.namelist() else ET.Element(W + "styles")
        styles = {s.get(W + "styleId"): s for s in styles_root.findall(W + "style")}
        numbering_root = _xml(archive.read("word/numbering.xml")) if "word/numbering.xml" in archive.namelist() else ET.Element(W + "numbering")
        numbering = _WordNumbering(numbering_root, styles)
        # Synthetic styles map inherited/custom/Chinese headings explicitly to HTML headings.
        heading_ids = {}
        for level in range(1, 7):
            sid = f"RibosomeCodeHeading{level}"
            while sid in styles:
                sid += "_"
            heading_ids[level] = sid
            style = ET.SubElement(styles_root, W + "style", {W + "type": "paragraph", W + "styleId": sid})
            ET.SubElement(style, W + "name", {W + "val": sid})
            ppr = ET.SubElement(style, W + "pPr")
            ET.SubElement(ppr, W + "outlineLvl", {W + "val": str(level - 1)})
        headings = []
        active_heading_nums = {}
        body = document.find(W + "body")
        for paragraph in body.iter(W + "p"):
            style_id, num_id, level, heading = _paragraph_properties(paragraph, styles)
            text = "".join(paragraph.xpath(".//w:t/text()", namespaces={"w": W[1:-1]}))
            # A paragraph-level list selection on a parent overrides the style's
            # list instance for descendants in the same abstract outline. Do not
            # merge counters globally: explicit list selections remain independent.
            direct_num = _val(paragraph.find(W + "pPr"), W + "numPr/" + W + "numId")
            if heading and direct_num is None and num_id not in {None, "0"}:
                ancestors = [depth for depth in active_heading_nums if depth < heading]
                parent_num = active_heading_nums[max(ancestors)] if ancestors else None
                if parent_num not in {None, "0"} and (
                    _val(numbering.nums.get(parent_num), W + "abstractNumId")
                    == _val(numbering.nums.get(num_id), W + "abstractNumId")
                ):
                    num_id = parent_num
            if num_id not in {None, "0"}:
                try:
                    level = numbering.advance(num_id, level, style_id)
                except ValueError:
                    if heading:
                        raise
                    continue
            if heading is None:
                continue
            active_heading_nums = {depth: value for depth, value in active_heading_nums.items()
                                   if depth < heading}
            active_heading_nums[heading] = num_id
            label = numbering.label(num_id, level) if num_id not in {None, "0"} else ""
            # A literal copy of the computed label must not be emitted twice.
            already_typed = bool(label) and bool(re.match(re.escape(label) + r"(?:\s|$)", text.lstrip()))
            if label and not already_typed:
                run = ET.Element(W + "r")
                ET.SubElement(run, W + "t", {XML_SPACE: "preserve"}).text = label + " "
                insertion = 1 if len(paragraph) and paragraph[0].tag == W + "pPr" else 0
                paragraph.insert(insertion, run)
                text = label + " " + text
            ppr = paragraph.find(W + "pPr")
            if ppr is None:
                ppr = ET.Element(W + "pPr")
                paragraph.insert(0, ppr)
            pstyle = ppr.find(W + "pStyle")
            if pstyle is None:
                pstyle = ET.Element(W + "pStyle")
                ppr.insert(0, pstyle)
            pstyle.set(W + "val", heading_ids[heading])
            for old in list(ppr.findall(W + "numPr")):
                ppr.remove(old)
            ET.SubElement(ET.SubElement(ppr, W + "numPr"), W + "numId", {W + "val": "0"})
            headings.append({"level": heading, "title": text})
        replacements = {
            "word/document.xml": ET.tostring(document, xml_declaration=True, encoding="UTF-8", standalone=True),
            "word/styles.xml": ET.tostring(styles_root, xml_declaration=True, encoding="UTF-8", standalone=True),
        }
        output = io.BytesIO()
        with ZipFile(output, "w") as target:
            target.comment = archive.comment
            for entry in archive.infolist():
                target.writestr(entry, replacements.pop(entry.filename, None) or archive.read(entry))
            for name, data in replacements.items():
                target.writestr(name, data)
    output.seek(0)
    style_map = "\n".join(f"p.{sid} => h{level}:fresh" for level, sid in heading_ids.items())
    return output, style_map, headings


## HTML tables, image assets, and checkpoints

Each top-level HTML heading starts a section; content before the first heading is an introduction. A document without headings has one section. Tables and lists are never split apart, even if they contain headings.

A hidden `.conversion-work/` folder in each document’s output folder stores `prepared.html`, extracted `img/` assets, section fragments, and `checkpoint.json`. The manifest records source SHA-256, pipeline/dependency versions, settings, section titles, fragment checksums, and conversion warnings. Files are flushed and replaced atomically after preparation and after every section. Interrupted preparation repeats; matching completed sections and prepared assets are reused after later interruptions.

Images retain their original formats, including EMF/WMF; some viewers cannot display these formats. Converter warnings are included in the report. Image filenames use content hashes, allowing assets to be published safely before replacing an older Markdown file.
HTML tables encode literal Markdown punctuation in cell text as HTML entities. This prevents Pandoc from interpreting text such as `1. instruction` as Markdown lists inside HTML and swallowing enclosing object wrappers. HTML readers recover the original cell text; merged/nested tables, inline formatting, images, and preformatted content are preserved. Structural line breaks keep the markup readable. Pipeline version 3 regenerates incompatible conversion fragments; existing final Markdown still requires `overwrite=True` to refresh.


In [ ]:
class HtmlTableMarkdownConverter(MarkdownConverter):
    """Keep the entire original table subtree as HTML, including nested tables."""

    def convert_table(self, el, text, parent_tags):
        # Pandoc's Markdown reader interprets Markdown inside raw HTML cells.
        # Literal "1. " text can become an unclosed list and swallow object Divs.
        # Encode Markdown punctuation as HTML entities in text nodes only; HTML
        # readers recover the exact text, while tags and image attributes stay HTML.
        def literal_text(value):
            if isinstance(value, NavigableString):
                punctuation = r"\`*_{}[]()#+-.!|~"
                return "".join(f"&#{ord(char)};" if char in punctuation else escape(char, quote=False)
                               for char in value)
            return escape(value, quote=False)

        # Structural whitespace stays outside cells, preserving inline/pre text.
        table = deepcopy(el)
        for container in [table, *table.find_all(["table", "thead", "tbody", "tfoot", "tr"])]:
            for child in list(container.contents):
                if isinstance(child, Tag):
                    child.insert_before("\n")
            container.append("\n")
        return "\n\n" + table.decode(formatter=HTMLFormatter(entity_substitution=literal_text)) + "\n\n"


def _digest(data):
    return hashlib.sha256(data).hexdigest()


def _atomic_bytes(path, data):
    """Flush a sibling temporary file, then replace the destination atomically."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary = tempfile.mkstemp(prefix=f".{path.name}.", dir=path.parent)
    try:
        with os.fdopen(descriptor, "wb") as stream:
            stream.write(data)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def _atomic_json(path, record):
    _atomic_bytes(path, json.dumps(record, ensure_ascii=False, indent=2).encode("utf-8"))


def _image_extension(content_type):
    known = {"image/png": ".png", "image/jpeg": ".jpg", "image/gif": ".gif",
             "image/svg+xml": ".svg", "image/x-emf": ".emf", "image/emf": ".emf",
             "image/x-wmf": ".wmf", "image/wmf": ".wmf", "image/tiff": ".tiff"}
    return known.get(content_type) or mimetypes.guess_extension(content_type) or ".bin"


def _split_sections(html):
    soup = BeautifulSoup(html, "html.parser")
    sections, nodes, title = [], [], "Introduction"
    for node in soup.contents:
        if isinstance(node, Tag) and re.fullmatch(r"h[1-6]", node.name):
            if "".join(nodes).strip():
                sections.append({"title": title, "html": "".join(nodes)})
            title, nodes = node.get_text(" ", strip=True), []
        nodes.append(str(node))
    if "".join(nodes).strip() or not sections:
        sections.append({"title": title, "html": "".join(nodes)})
    return sections


def _signature(source_bytes):
    return {
        "source_sha256": _digest(source_bytes),
        "pipeline_version": PIPELINE_VERSION,
        "dependencies": {name: version(name) for name in ("mammoth", "markdownify", "beautifulsoup4", "lxml")},
        "settings": {"heading_style": "ATX", "tables": "html", "images": "relative", "sections": "headings"},
    }


def _read_checked(path, expected):
    data = path.read_bytes()
    if _digest(data) != expected:
        raise ValueError(f"Checkpoint checksum mismatch: {path}")
    return data


def _load_prepared(work, signature):
    """Invalid/incomplete preparation is rebuilt; incomplete sections are retried."""
    try:
        manifest = json.loads((work / "checkpoint.json").read_text(encoding="utf-8"))
        if manifest["signature"] != signature:
            return None
        warnings = manifest["warnings"]
        if not isinstance(warnings, list) or any(
            not isinstance(warning, dict)
            or not isinstance(warning.get("type"), str)
            or not isinstance(warning.get("message"), str)
            for warning in warnings
        ):
            return None
        html = _read_checked(work / "prepared.html", manifest["html_sha256"]).decode("utf-8")
        for asset in manifest["assets"]:
            relative = Path(asset["path"])
            if relative.parent != Path("img") or relative.name != asset["path"].split("/")[-1]:
                return None
            _read_checked(work / relative, asset["sha256"])
        sections = _split_sections(html)
        metadata = [{"title": s["title"], "html_sha256": _digest(s["html"].encode("utf-8"))} for s in sections]
        if metadata != manifest["sections"] or not isinstance(manifest["completed"], dict):
            return None
        return manifest, sections
    except (OSError, ValueError, KeyError, TypeError, AttributeError):
        return None


def _prepare_document(source_bytes, work, signature):
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
    numbered, style_map, headings = prepare_numbered_docx(source_bytes)
    assets = {}

    @mammoth.images.img_element
    def save_image(image):
        with image.open() as stream:
            data = stream.read()
        digest = _digest(data)
        relative = f"img/{digest}{_image_extension(image.content_type)}"
        _atomic_bytes(work / relative, data)
        assets[relative] = {"path": relative, "sha256": digest}
        return {"src": relative}

    result = mammoth.convert_to_html(numbered, style_map=style_map, convert_image=save_image,
                                     include_embedded_style_map=False)
    html = result.value
    sections = _split_sections(html)
    _atomic_bytes(work / "prepared.html", html.encode("utf-8"))
    manifest = {
        "signature": signature,
        "html_sha256": _digest(html.encode("utf-8")),
        "assets": list(assets.values()),
        "headings": headings,
        "sections": [{"title": s["title"], "html_sha256": _digest(s["html"].encode("utf-8"))} for s in sections],
        "completed": {},
        "warnings": [{"type": m.type, "message": m.message} for m in result.messages],
    }
    _atomic_json(work / "checkpoint.json", manifest)
    return manifest, sections


def _valid_fragments(work, manifest, count):
    valid = {}
    for index in range(count):
        expected = manifest["completed"].get(str(index))
        if not isinstance(expected, str):
            continue
        try:
            _read_checked(work / "sections" / f"{index:06d}.md", expected)
        except (OSError, ValueError):
            continue
        valid[str(index)] = expected
    return valid


def _publish_document(target, work, manifest, count):
    # Content-addressed assets are safe to publish before replacing an older Markdown file.
    for asset in manifest["assets"]:
        _atomic_bytes(target.parent / asset["path"], _read_checked(work / asset["path"], asset["sha256"]))
    fragments = [_read_checked(work / "sections" / f"{i:06d}.md", manifest["completed"][str(i)]).decode("utf-8").strip()
                 for i in range(count)]
    markdown = "\n\n".join(fragment for fragment in fragments if fragment).rstrip() + "\n"
    _atomic_bytes(target, markdown.encode("utf-8"))


## Resumable conversion and two progress bars

The outer bar counts files; the inner bar counts sections and shows the current numbered heading. During initial DOCX preparation the inner bar shows an unknown total. On resume it starts at the number of verified section fragments already saved.

`resume=True` reuses compatible checkpoints. Changed source bytes or conversion versions/settings invalidate partial work. Missing or damaged fragments are regenerated. `resume=False` starts partial work again. Existing final Markdown is skipped unless `overwrite=True`, including when its source has changed; use `overwrite=True` to refresh those outputs.

A normal exception records a failed file and continues the batch. A kernel interrupt stops processing immediately and keeps checkpoints. Rerun the batch cell to resume. After every section is complete, the notebook publishes image assets, atomically replaces the final Markdown, and removes temporary checkpoint data. Use one notebook process per output tree to avoid competing writers.

In [ ]:
def convert_docx_to_markdown(source, target, *, resume=True, overwrite=False, show_progress=True):
    """Convert one DOCX; restart safely from a matching section checkpoint."""
    source, target = Path(source).expanduser().resolve(), Path(target).expanduser().resolve()
    if not source.is_file():
        raise FileNotFoundError(source)
    if source == target or target.suffix.lower() != ".md":
        raise ValueError("target must be a .md file different from source")
    if target.is_file() and not overwrite:
        return {"source": source, "target": target, "status": "skipped", "warnings": [], "resumed_sections": 0}
    work = target.parent / ".conversion-work"
    with tqdm(total=None, desc=f"Preparing document: {source.name}", unit="section", position=1,
              leave=False, disable=not show_progress) as bar:
        source_bytes = source.read_bytes()
        signature = _signature(source_bytes)
        prepared = _load_prepared(work, signature) if resume else None
        if prepared is None:
            prepared = _prepare_document(source_bytes, work, signature)
        manifest, sections = prepared
        manifest["completed"] = _valid_fragments(work, manifest, len(sections))
        resumed = len(manifest["completed"])
        bar.reset(total=len(sections))
        bar.update(resumed)
        converter = HtmlTableMarkdownConverter(heading_style="ATX", bullets="-", strip_document="strip")
        for index, section in enumerate(sections):
            if str(index) in manifest["completed"]:
                continue
            bar.set_description(f"{source.name}: {section['title'][:80]}")
            fragment = converter.convert(section["html"]).strip().encode("utf-8")
            _atomic_bytes(work / "sections" / f"{index:06d}.md", fragment)
            manifest["completed"][str(index)] = _digest(fragment)
            _atomic_json(work / "checkpoint.json", manifest)
            bar.update(1)
        bar.set_description(f"Publishing: {source.name}")
        _publish_document(target, work, manifest, len(sections))
        shutil.rmtree(work)
    return {"source": source, "target": target, "status": "converted", "warnings": manifest["warnings"],
            "resumed_sections": resumed, "sections": len(sections), "images": len(manifest["assets"])}


def convert_code_docs(source_root, output_root=None, *, resume=True, overwrite=False, show_progress=True):
    """Convert one root or a list of roots; default outputs live in each root/.md."""
    if isinstance(source_root, (list, tuple)):
        if not source_root:
            raise ValueError("source_root must contain at least one directory")
        roots = [Path(path).expanduser().resolve() for path in source_root]
        if output_root is None:
            outputs = [root / ".md" for root in roots]
        elif isinstance(output_root, (list, tuple)) and len(output_root) == len(roots):
            outputs = [Path(path).expanduser().resolve() for path in output_root]
        else:
            raise ValueError("For a list of source roots, output_root must be None or a matching list")
        # Validate all destinations before publishing anything, including cross-root
        # overlap that could let one conversion overwrite another root's inputs.
        pairs = list(dict.fromkeys(zip(roots, outputs)))
        if len({output for _, output in pairs}) != len(pairs):
            raise ValueError("Source roots must have distinct output directories")
        for root, output in pairs:
            if not root.is_dir():
                raise NotADirectoryError(root)
            if any(other.is_relative_to(output) for other in roots):
                raise ValueError("output_root must not be a source root or its ancestor")
        reports = [convert_code_docs(root, output, resume=resume, overwrite=overwrite,
                                     show_progress=show_progress) for root, output in pairs]
        return {"root": [root for root, _ in pairs],
                "output_root": [output for _, output in pairs],
                "discovered": sum(report["discovered"] for report in reports),
                **{key: [item for report in reports for item in report[key]]
                   for key in ("converted", "skipped", "failed", "warnings")},
                "reports": reports}
    root = Path(source_root).expanduser().resolve()
    if not root.is_dir():
        raise NotADirectoryError(root)
    output = Path(output_root).expanduser().resolve() if output_root is not None else root / ".md"
    if root.is_relative_to(output):
        raise ValueError("output_root must not be the source root or its ancestor")
    # Exclude the output tree so a custom output directory cannot become an input.
    sources = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() == ".docx"
                     and not p.name.startswith("~$") and not p.is_relative_to(output))
    report = {"root": root, "output_root": output, "discovered": len(sources),
              "converted": [], "skipped": [], "failed": [], "warnings": []}
    targets = {}
    for source in sources:
        stem = source.stem.rstrip(" .") or "_"
        target = output / source.relative_to(root).parent / stem / f"{stem}.md"
        targets.setdefault(target, []).append(source)
    with tqdm(total=len(sources), desc="DOCX files", unit="file", position=0,
              leave=True, disable=not show_progress) as bar:
        for target, candidates in targets.items():
            for source in candidates:
                try:
                    if len(candidates) > 1:
                        raise ValueError(f"Multiple DOCX files map to the same output: {target}")
                    result = convert_docx_to_markdown(source, target, resume=resume,
                                                       overwrite=overwrite, show_progress=show_progress)
                    report[result["status"]].append(result)
                    report["warnings"].extend({"source": source, **warning} for warning in result["warnings"])
                except Exception as error:
                    report["failed"].append({"source": source, "target": target,
                                              "error": f"{type(error).__name__}: {error}"})
                    if show_progress:
                        tqdm.write(f"Failed: {source}: {error}")
                finally:
                    bar.update(1)
                    bar.set_postfix(converted=len(report["converted"]), skipped=len(report["skipped"]),
                                    failed=len(report["failed"]))
    if show_progress:
        tqdm.write(f"Finished: {len(report['converted'])} converted, {len(report['skipped'])} skipped, "
                   f"{len(report['failed'])} failed, {len(report['warnings'])} warning(s).")
    return report


## Configure and run

Outputs follow `.md/<relative source folders>/<document>/<document>.md`, with images in the adjacent `img/` folder. Word lock files (`~$...`) are ignored. Duplicate output destinations are reported as failures. Set `SHOW_PROGRESS=False` for a quiet programmatic run. The returned `processing_report` includes paths, counts, resumed section counts, warnings, and errors.

For a single file, call `convert_docx_to_markdown(source, target, resume=True, overwrite=False)`; `target` should have its own document folder.
`SOURCE_ROOT` accepts one path or a list of paths. With `OUTPUT_ROOT=None`, each root writes to its own `.md` folder. To customize destinations for a list, set `OUTPUT_ROOT` to a matching list of paths. The combined report retains the counts and result lists above, plus per-root `reports`. Duplicate source/output pairs are processed once.


In [ ]:
SOURCE_ROOT = [Path("/d/devel/rag/ribosome/assets/code-docs.4.0"),
               Path("/d/devel/rag/ribosome/assets/code-docs.5.0")]
OUTPUT_ROOT = None  # Each source root writes to its own .md folder.
RESUME = True
OVERWRITE = False
SHOW_PROGRESS = True


In [ ]:
# | notest
processing_report = convert_code_docs(
    SOURCE_ROOT,
    OUTPUT_ROOT,
    resume=RESUME,
    overwrite=OVERWRITE,
    show_progress=SHOW_PROGRESS,
)
{
    "discovered": processing_report["discovered"],
    "converted": len(processing_report["converted"]),
    "skipped": len(processing_report["skipped"]),
    "failed": len(processing_report["failed"]),
    "warnings": len(processing_report["warnings"]),
    "resumed_sections": sum(item["resumed_sections"] for item in processing_report["converted"]),
    "failures": processing_report["failed"],
}
